# Embedding-dimension sweep: GUI screen prediction vs. projection size

Reads the output of `4_sweep_embedding_dimensions.py` from `dimension_sweep/` and presents it the way the other result notebooks do: one compact table and one paper-style figure per sweep configuration (protocol × projector hidden width), with the Screen2Vec Table 3 reference and a LaTeX fragment exported to `graphics/dimension_sweep/`.

Nothing is trained or evaluated here. Run the sweep first on a machine with the raw embeddings, for example

```bash
python 4_sweep_embedding_dimensions.py run --gpu 0                      # permissive recipe, seeds 42
python 4_sweep_embedding_dimensions.py run --protocol strict --dims 64,256,768
python 4_sweep_embedding_dimensions.py run --dims 128,768 --seeds 42,43,44   # error bars
```

then execute this notebook. Every `results*.json` under `dimension_sweep/` is picked up automatically.

**Reading the numbers** (`FINDINGS.md` §2–3): **Top-1** and **MRR** are the informative columns. Top-0.1 % and above are saturated by the concatenated app-description half (an app-only oracle already reaches 0.9968 at Top-0.1 %), and normalised RMSE is not comparable to the paper's because the paper evaluates unnormalised vectors. The paper reference row is Screen2Vec-EUCLIDEAN-spatial, 768 learned + 768 app-description dimensions; every sweep point uses D learned + 768 app-description dimensions.

In [ ]:
from pathlib import Path
import importlib
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "utils.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository or a subdirectory.")
sys.path.insert(0, str(ROOT))
sweep = importlib.import_module("4_sweep_embedding_dimensions")

SWEEP_DIR = ROOT / "dimension_sweep"
PDF_DIR = ROOT / "graphics/dimension_sweep"
PDF_DIR.mkdir(parents=True, exist_ok=True)

FIGSIZE = (4.75, 2.75)          # inches, paper column; change only this for sizing
SHOW_SATURATED_PANEL = True     # second panel with the Top-k% columns
TABLE_METRICS = ["top1", "mrr", "top001pct", "top01pct", "top1pct", "nrmse"]
LABELS = dict(sweep.METRIC_LABELS)

# Paper reference: read from the evaluator when its dependencies are installed,
# otherwise fall back to the same numbers (Screen2Vec Table 3, EUCLIDEAN-spatial).
try:
    PAPER = importlib.import_module("4_evaluate_screen_prediction").PAPER_TABLE3["Screen2Vec-EUCLIDEAN-spatial"]
except Exception:
    PAPER = {"top1": 0.061, "top001pct": 0.258, "top01pct": 0.969, "top1pct": 0.998, "top5pct": 1.00, "nrmse": 0.853}

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150, "pdf.fonttype": 42, "ps.fonttype": 42,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 180)

## 1. Load every sweep configuration

One results file per protocol and hidden width, e.g. `results.json` (permissive, hidden 1024), `results_strict.json`, `results_h4096.json`. Each holds one entry per (dimension, seed).

In [ ]:
result_files = sorted(p for p in SWEEP_DIR.glob("results*.json") if p.is_file())
if not result_files:
    raise FileNotFoundError(f"No results*.json under {SWEEP_DIR}; run 4_sweep_embedding_dimensions.py first.")

configs = {}
for path in result_files:
    data = json.loads(path.read_text(encoding="utf-8"))
    rows = []
    for dim_key, by_seed in data.get("runs", {}).items():
        for seed_key, m in by_seed.items():
            row = {"proj_dim": int(dim_key), "seed": int(seed_key)}
            row.update({k: m.get(k, np.nan) for k in sweep.METRIC_KEYS})
            row.update({k: m.get(k) for k in sweep.POPULATION_KEYS})
            row["completed_at"] = m.get("completed_at")
            rows.append(row)
    if not rows:
        continue
    name = path.stem.removeprefix("results").lstrip("_") or "permissive"
    configs[name] = {"meta": data.get("meta", {}), "runs": pd.DataFrame(rows).sort_values(["proj_dim", "seed"])}

overview = pd.DataFrame([{
    "config": name,
    "protocol": c["meta"].get("protocol"),
    "hidden_dim": c["meta"].get("hidden_dim"),
    "dims": ", ".join(str(d) for d in sorted(c["runs"]["proj_dim"].unique())),
    "seeds": ", ".join(str(s) for s in sorted(c["runs"]["seed"].unique())),
    "runs": len(c["runs"]),
    "git": c["meta"].get("git_revision"),
    "trace index coverage": (c["meta"].get("trace_index_meta") or {}).get("coverage"),
} for name, c in configs.items()])
display(overview.style.format({"trace index coverage": "{:.4f}"}, na_rep="–").hide(axis="index"))

## 2. Summary per configuration

Mean over seeds (± sample standard deviation when a dimension was repeated). The last column gives the Top-1 gain over the paper row; the smallest dimension whose mean Top-1 beats the paper is marked. Full-precision values are exported as CSV, the compact table as a LaTeX fragment.

In [ ]:
def summarise(runs):
    g = runs.groupby("proj_dim")
    out = pd.DataFrame({"n": g["seed"].size()})
    for k in TABLE_METRICS:
        out[f"{k}_mean"] = g[k].mean()
        out[f"{k}_std"] = g[k].std(ddof=1).fillna(0.0)
    out["top1_delta_vs_paper"] = out["top1_mean"] - PAPER["top1"]
    out["beats_paper_top1"] = out["top1_mean"] > PAPER["top1"]
    return out.reset_index()


def compact(summary):
    rows = []
    for r in summary.itertuples(index=False):
        row = {"D": int(r.proj_dim), "n": int(r.n)}
        for k in TABLE_METRICS:
            mean, std = getattr(r, f"{k}_mean"), getattr(r, f"{k}_std")
            row[LABELS[k]] = f"{mean:.4f} ± {std:.4f}" if r.n > 1 else f"{mean:.4f}"
        row["Δ Top-1 vs paper"] = f"{r.top1_delta_vs_paper:+.4f}"
        rows.append(row)
    ref = {"D": "768 (paper)", "n": ""}
    ref.update({LABELS[k]: (f"{PAPER[k]:.4f}" if k in PAPER else "–") for k in TABLE_METRICS})
    ref["Δ Top-1 vs paper"] = ""
    rows.append(ref)
    return pd.DataFrame(rows)


def latex_fragment(summary, name):
    cols = " & ".join(LABELS[k] for k in TABLE_METRICS)
    lines = [
        r"\begin{table}[t]", r"    \centering",
        rf"    \caption{{GUI screen prediction against the learned screen-embedding dimension $D$ "
        rf"(sweep configuration \texttt{{{name}}}; every point uses $D$ + 768 app-description dimensions). "
        r"Mean over seeds, $\pm$ sample SD where repeated. Top-1 and MRR are the informative columns; "
        r"Top-0.1\,\% and above are saturated by app identity and normalised RMSE is not comparable to the paper's.}",
        rf"    \label{{tab:dimension_sweep_{name}}}", r"    \small",
        rf"    \begin{{tabular}}{{@{{}}rr{'r' * len(TABLE_METRICS)}@{{}}}}", r"        \toprule",
        rf"        $D$ & $n$ & {cols} \\", r"        \midrule",
    ]
    for r in summary.itertuples(index=False):
        cells = []
        for k in TABLE_METRICS:
            mean, std = getattr(r, f"{k}_mean"), getattr(r, f"{k}_std")
            cells.append(f"{mean:.4f} $\\pm$ {std:.4f}" if r.n > 1 else f"{mean:.4f}")
        lines.append(f"        {int(r.proj_dim)} & {int(r.n)} & " + " & ".join(cells) + r" \\")
    lines += [r"        \midrule",
              "        768 (paper) & -- & " + " & ".join(f"{PAPER[k]:.4f}" if k in PAPER else "--" for k in TABLE_METRICS) + r" \\",
              r"        \bottomrule", r"    \end{tabular}", r"\end{table}"]
    return "\n".join(lines)


summaries = {}
for name, c in configs.items():
    summary = summarise(c["runs"])
    summaries[name] = summary
    summary.to_csv(PDF_DIR / f"dimension_sweep_{name}.csv", index=False)
    (PDF_DIR / f"dimension_sweep_{name}.tex").write_text(latex_fragment(summary, name) + "\n", encoding="utf-8")
    winners = summary.loc[summary["beats_paper_top1"], "proj_dim"]
    note = (f"smallest D beating the paper's Top-1 ({PAPER['top1']:.4f}): **{int(winners.min())}**"
            if len(winners) else f"no dimension beats the paper's Top-1 ({PAPER['top1']:.4f})")
    display(Markdown(f"### `{name}` — protocol {c['meta'].get('protocol')}, hidden {c['meta'].get('hidden_dim')} · {note}"))
    display(compact(summary).style.hide(axis="index"))

## 3. Curves

Left: Top-1 and MRR against $D$ on a log₂ axis, dashed line = paper Top-1, error bars = sample SD over seeds where repeated. Right (optional): the Top-k % columns, shown only to document their saturation. One PDF per configuration goes to `graphics/dimension_sweep/`.

In [ ]:
def plot_config(name, summary, meta):
    dims = summary["proj_dim"].to_numpy()
    multi = (summary["n"] > 1).any()
    n_panels = 2 if SHOW_SATURATED_PANEL else 1
    fig, axes = plt.subplots(1, n_panels, figsize=(FIGSIZE[0] * n_panels, FIGSIZE[1]), squeeze=False)
    axes = axes[0]

    def curve(ax, key, color, label):
        ax.errorbar(dims, summary[f"{key}_mean"], yerr=summary[f"{key}_std"] if multi else None,
                    color=color, marker="o", markersize=3.5, linewidth=1.4, capsize=2.5, label=label)
        if key in PAPER:
            ax.axhline(PAPER[key], color=color, linestyle=(0, (4, 3)), linewidth=1, alpha=0.7)

    for ax in axes:
        ax.set_xscale("log", base=2)
        ax.set_xticks(dims)
        ax.set_xticklabels([str(d) for d in dims], fontsize=7)
        ax.minorticks_off()
        ax.set_xlabel("Screen-embedding dimension D (of D + 768)")
        ax.tick_params(labelsize=7)

    curve(axes[0], "top1", "#4C78A8", "Top-1")
    curve(axes[0], "mrr", "#E45756", "MRR")
    axes[0].set_ylim(0, None)
    axes[0].set_ylabel("Retrieval quality")
    axes[0].legend(frameon=False, fontsize=7, loc="lower right", title="dashed: paper Top-1", title_fontsize=7)

    if SHOW_SATURATED_PANEL:
        curve(axes[1], "top001pct", "#54A24B", "Top-0.01%")
        curve(axes[1], "top01pct", "#F58518", "Top-0.1%")
        curve(axes[1], "top1pct", "#72B7B2", "Top-1%")
        axes[1].set_ylim(0, 1.02)
        axes[1].set_ylabel("Accuracy (saturated by app identity)")
        axes[1].legend(frameon=False, fontsize=7, loc="lower right")

    fig.tight_layout()
    fig.savefig(PDF_DIR / f"dimension_sweep_{name}.pdf", bbox_inches="tight")
    fig.savefig(PDF_DIR / f"dimension_sweep_{name}.png", bbox_inches="tight")
    display(Markdown(f"**`{name}`** — protocol {meta.get('protocol')}, hidden {meta.get('hidden_dim')}"))
    plt.show()


for name, summary in summaries.items():
    plot_config(name, summary, configs[name]["meta"])

## 4. Individual runs and provenance

Every (dimension, seed) run with its evaluated population, so a suspicious point can be traced to its log in `dimension_sweep/logs/`.

In [ ]:
for name, c in configs.items():
    display(Markdown(f"**`{name}`**"))
    cols = ["proj_dim", "seed", "top1", "mrr", "top001pct", "top01pct", "top1pct", "top5pct", "nrmse",
            "n_traces", "n_screens_in_traces", "n_instances", "completed_at"]
    display(c["runs"][cols].style.format({k: "{:.4f}" for k in sweep.METRIC_KEYS}, na_rep="–").hide(axis="index"))
    meta = {k: v for k, v in c["meta"].items() if k != "trace_index_meta"}
    meta["trace_index_coverage"] = (c["meta"].get("trace_index_meta") or {}).get("coverage")
    print(json.dumps(meta, indent=2))

print("\nfiles in", PDF_DIR.relative_to(ROOT))
for path in sorted(PDF_DIR.iterdir()):
    print(f"  {path.name:45s} {path.stat().st_size / 1024:8.1f} KB")